<a href="https://colab.research.google.com/github/Aryaniitkgp/ComVis/blob/Transformer/ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [2]:
class PatchEmbedding(nn.Module):
  def __init__(self,img_size=224,patch_size=16,inchannels=3,emd_dim=768):
    super().__init__()
    self.patch_size=patch_size
    self.num_patches=(img_size//patch_size)**2

    self.prejection=nn.Conv2d(inchannels,emd_dim,kernel_size=patch_size,stride=patch_size)
    self.cls_token=nn.Parameter(torch.zeros(1,1,emd_dim))
    self.pos_embedding=nn.Parameter(torch.zeros(1,1+self.num_patches,emd_dim))
  def forward(self,x):
    B=x.shape[0]
    x=self.prejection(x).flatten(2)
    x=x.transpose(1,2)
    cls_tokens=self.cls_token.expand(B,-1,-1)
    x=torch.cat((cls_tokens,x),dim=1)
    x=x+self.pos_embedding
    return x

In [3]:
class MLP(nn.Module):
  def __init__(self,in_features,hidden_features,out_features):
    super().__init__()
    self.fc1=nn.Linear(in_features,hidden_features)
    self.act=nn.GELU()
    self.fc2=nn.Linear(hidden_features,out_features)
  def forward(self, x):
    x=self.fc1(x)
    x=self.act(x)
    x=self.fc2(x)
    return x

In [4]:
class TranformerEncoderBlock(nn.Module):
  def __init__(self, emd_dim=768,num_head=12, mlp_ratio=4.0):
    super().__init__()
    self.norm1=nn.LayerNorm(emd_dim)
    self.attn=nn.MultiheadAttention(emd_dim,num_head, batch_first=True)
    self.norm2=nn.LayerNorm(emd_dim)
    hidden_features=int(emd_dim*mlp_ratio)
    self.mlp=MLP(emd_dim,hidden_features,emd_dim)
  def forward(self,x):
    norm_x=self.norm1(x)
    attn_output,_=self.attn(norm_x,norm_x,norm_x)
    x=x+attn_output
    x=x+self.mlp(self.norm2(x))
    return x


In [5]:
class ViT(nn.Module):
  def __init__(self,img_size=224,patch_size=16,in_channel=3,num_classes=1000,emd_dim=768,depth=12,num_head=12,mlp_ratio=4.0):
    super().__init__()
    self.patch_embed=PatchEmbedding(img_size,patch_size,in_channel,emd_dim)
    self.blocks=nn.ModuleList([TranformerEncoderBlock(emd_dim,num_head,mlp_ratio) for _ in range(depth)])
    self.norm=nn.LayerNorm(emd_dim)
    self.head=nn.Linear(emd_dim,num_classes)
  def forward(self,x):
    x=self.patch_embed(x)
    for block in self.blocks:
      x=block(x)
    x=self.norm(x)
    cls_token_output=x[:,0]
    out=self.head(cls_token_output)
    return out

In [6]:
if __name__ =="__main__":
  dummy_img=torch.randn(2,3,224,224)
  model=ViT(img_size=224,patch_size=16,num_classes=10,emd_dim=768,depth=12,num_head=12)
  prediction=model(dummy_img)
  print(f'input image shape{dummy_img.shape}')
  print(f'output image shape{prediction.shape}')

input image shapetorch.Size([2, 3, 224, 224])
output image shapetorch.Size([2, 10])
